# Week 09: More Text

We've seen how to do some basic text manipulation and transformations.

But, what if we have to extract more complex patterns from text, or want to parse structured text files ?

In [ ]:
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/data_utils.py

In [ ]:
import re
import requests

from bs4 import BeautifulSoup

from data_utils import image_from_url

## Revisit something from WK01

In the very first lecture we briefly saw an example of how to download images by accessing websites through code.

Let's break this down into steps and see how we can use code to "read" websites.

### Requests

The internet is a gigantic, connected, hard drive. When we access an `url` with our browser, what we're actually doing is downloading a file from that location, and interpreting its results as a webpage.

But the file is just an `html` text file. We can download and print it.

In [ ]:
res = requests.get("https://www.newschool.edu/")

print(res.text)

## `HTML` / `XML`

`HTML` and `XML` are $2$ formats of structured text files.

They use nested _tags_ (`<tag>content</tag>`) to contextualize or give specific meaning to different parts of the text file.

These tags in `HTML`, for example, are used to specify an image, a link and a list, respectively:

```html
<img src="image.jpg"/>

<a href="https://www.newschool.edu/">Link Text</a>

<ul>
  <li>List Item 1</li>
  <li>List Item 2</li>
</ul>
```

`XML` is a very similar format, but instead of specifying elements to be displayed, it's mainly used for storing and transferring structured text. It's not unlike `JSON`, but focused on text information:

```xml
<document>
  <book>
    <title>Towards a Philosophy of Photography</title>
    <author>Vilém Flusser</author>
    <year>1984</year>
  </book>
</document>
```

In [ ]:
# TODO: try other URLs, try to find image tags <img> and link tags <a>

tns_txt = res.text

for line in tns_txt.split("\n"):
  if "<img" in line:
    print(line)

## Why do we care?

Everything is on the internet.

If we know how to automate information retrieval and extraction from urls, we can create our own datasets by carefully processing and combining data from different sources.

For example, maybe we want to create a dataset of news articles about the environment.

We can get a list of recent articles from these newspapers by accessing their RSS feeds (`xml`) or `html` pages:
- https://www.theguardian.com/environment/rss
- https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml
- https://www.latimes.com/environment/rss2.0.xml
- https://www.washingtonpost.com/climate-environment/
- https://www.chinadaily.com.cn/china/environment
- https://timesofindia.indiatimes.com/rssfeeds/2647163.cms
- https://english.elpais.com/climate/
- https://japannews.yomiuri.co.jp/science-nature/climate-change/

Let's try to get a list of article titles, links and images from one of these.

The `split()` and `strip()` functions might be useful, as well as the `in` operator for checking if a specific text is present _in_ a line.

In [ ]:
# TODO: get list of titles, links and images from one of the newspaper feeds

nyt_res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml")
nyt_xml = nyt_res.text.split("\n")

all_titles = []
for line in nyt_xml:
  if "<title>" in line:
    a_title = line.replace("<title>", "").replace("</title>", "")
    print(a_title.strip())

In [ ]:
# TODO: get list of titles, links and images from one of the newspaper feeds

nyt_res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml")
nyt_xml = nyt_res.text.split("\n")

all_titles = []
for line in nyt_xml:
  if "<media:content" in line:
    https_idx = line.find("https")
    jpg_idx = line.find("jpg")
    if https_idx > -1 and jpg_idx > -1:
      img_url = line[https_idx:jpg_idx+3]
      img = image_from_url(img_url)
      display(img)

In [ ]:
# experiment with find()

line = '<media:content height="1800" medium="image" url="https://static01.nyt.com/images/2026/04/02/multimedia/02cli-microplastics-gcwf/02cli-microplastics-gcwf-mediumSquareAt3X.jpg" width="1800"/>'

line.find("jpg")

## A good start

Going through an `html` or `xml` file line-by-line is ok. We can quickly get a sense of the document's structure and see how many articles we get, but if we actually want to extract the link, title and image of an article, while keeping that information together... it gets kind of messy.

In [ ]:
res = requests.get("https://www1.folha.uol.com.br/ambiente/")
res.encoding = "utf8"
fsp_climate = res.text

# number of characters in the file
print(len(fsp_climate), "characters\n")

# find "c-newslist__title" and "c-most-read__section" to isolate actual list of articles
list0_idx = fsp_climate.find("c-newslist__title")
list1_idx = fsp_climate.find("c-most-read__section")

# only go through the list of articles (skipping headlines 🤦🤷)
fsp_climate_main = fsp_climate[list0_idx:list1_idx]
fsp_climate_lines = fsp_climate_main.split("\n")
print(len(fsp_climate_main), "characters\n")

# use the "class" attribute of specific tags to isolate lines of interest
for line in fsp_climate_lines[:1050]:
  if "c-image-aspect-ratio__link" in line:
    print("  url: ", line.strip())
  elif "c-lazyload" in line:
    print("image: ", line.strip())
  elif "c-headline__title" in line:
    print("title: ", line.strip())

### Try with another link

Try to do something similar to extract article titles, links and images from another newspaper.

The `xml` feeds are a bit easier, but still... this is ***HARD***.

Each page has its own formatting and structuring, and even if we isolate the lines with the content we want, like we did above, we still haven't extracted the titles, images or links, only the `html` or `xml` with that information.

In [ ]:
# TODO: try to get titles, links and images from one of the other links

In [ ]:
# TODO: get images using regex

nyt_res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/Climate.xml")
nyt_xml = nyt_res.text.split("\n")

all_titles = []
for line in nyt_xml:
  if "<media:content" in line:
    img_url = re.findall(r"(https://.+(jpg|png|gif|webm))", line)[0][0]
    print(img_url)
    # img = image_from_url(img_url)
    # display(img)

## RegEx to the RegEx-cue

Instead of trying to come up with specific for loops and parsing logic to find the items we're interested in, we can use _regular expressions_ to describe more sophisticated patterns to extract.

For example, we can find and extract all instances of the pattern `<img`, with:
```
re.findall(r"<img", txt)
```

In [ ]:
# match all <img
print(re.findall(r"<img", fsp_climate))

# match all src=
print(re.findall(r"src=", fsp_climate))

## 🤔

This is almost useful.

In order to fully unlock all of its powers, we have to learn a few other regular expression commands and modifiers.

### Character Classes
- `.`: Any character except a newline.
- `\d`: Any digit (0-9).
- `\D`: Any non-digit character.
- `\w`: Any word character (alphanumeric plus underscore).
- `\W`: Any non-word character.
- `\s`: Any whitespace character.
- `\S`: Any non-whitespace character.
- `[abc]`: Matches any one character in the set 'a', 'b', or 'c'.
- `[^abc]`: Matches any character not in the set 'a', 'b', or 'c'.
- `[a-z]`: Matches any lowercase letter in the specified range.

### Anchors
- `^`: Matches the start of the string (or start of a line with the re.MULTILINE flag).
- `$`: Matches the end of the string (or end of a line with the re.MULTILINE flag).

### Quantifiers
- `*`: Zero or more repetitions of the preceding element.
- `+`: One or more repetitions of the preceding element.
- `?`: Zero or one repetition of the preceding element.
- `{n}`: Exactly n repetitions.
- `{n,}`: At least n repetitions.
- `{n,m}`: Between n and m repetitions.
- `*?`, `+?`, `??`, `{n,m}?`: Non-greedy versions of the quantifiers, matching as few repetitions as possible.

### Tip

We can help the regex engine and simplify our patterns by removing newlines and extra spaces.

the following code replaces any repeating sequence of whitespace characters with a single space.

In [ ]:
fsp_climate_line = re.sub(r"\s+", " ", fsp_climate)

## RegEx for real now

Let's extract clean titles, links and images from the articles.

In [ ]:
# match all src= followed by anything followed by a space
# this matches everything between the first "src" in the file until the very last ">" symbol
display(re.findall(r"src.+>", fsp_climate_line)[:4])

# match all src followed by anything until the first ">"
display(re.findall(r"src.+?>", fsp_climate_line)[:4])

# match all "headline content" <div> blocks
display(re.findall(r"<div[^>]+?headline__content.+?</div>", fsp_climate_line)[:4])

In [ ]:
# use parentheses grouping to match and single out href blocks within the "headline content" <div> blocks
href_pattern = r"<a.+?href=\"(.+?)\""
pattern = fr"<div[^>]+?headline__content.+?{href_pattern}.+?</div>"
display(re.findall(pattern, fsp_climate_line)[:4])

# use 2 parentheses groups to match and single out href and h2 blocks within the "headline content" <div> blocks
h2_pattern = r"<h2.+?>\s*(.+?)\s*</h2>"
link_title_pattern = fr"<div[^>]+?headline__content.+?{href_pattern}.+?{h2_pattern}.+?</div>"
links_titles = re.findall(link_title_pattern, fsp_climate_line)
display(links_titles[:4])

# use parentheses grouping to match and single out img and data-src blocks within the "headline media" <div> blocks
img_pattern = r"<img.+?data-src=\"(.+?)\".+?>"
link_img_pattern = fr"<div[^>]+?headline__media-wrapper.+?{href_pattern}.+?{img_pattern}.+?</div>"
links_imgs = re.findall(link_img_pattern, fsp_climate_line)
display(links_imgs[:4])

### Clean up

We have links and titles in one list, and links and images in another.

We can use the links to make sure all our articles have all three properties.

In [ ]:
link2info = { link: { "link": link, "title": title } for link, title in links_titles }

for link,img in links_imgs:
  if link in link2info:
    link2info[link]["img"] = img
  else:
    raise(Exception(f"missing: {link}"))

list(link2info.values())[:4]

## Exercise

Use RegEx patterns to get urls and download images from a google/bing image search.

The `image_from_url()` function imported above can be used to turn an image `url` into a `PIL` image.

### The Search

The following are urls for doing image searches on google and bing:

In [ ]:
QUERY_TERM = "gudetama"

BING_URL = f"https://www.bing.com/images/search?q={QUERY_TERM}"
GOOGLE_URL = f"https://www.google.com/search?udm=2&q={QUERY_TERM}"

In [ ]:
# TODO: Extract image urls and display images

## `HTML`/`XML` Selectors

RegEx is good and efficient, but can get a little confusing at times.

Depending on the structure of the `html`/`xml` file we're trying to parse, it might be useful to use a library that can traverse the file using tag names, attributes and hierarchical structure.

For example, consider the following `html`/`xml` file structures:

```xml
<tag attribute="value" id="unique-name" class="list of classes">
  content
</tag>
```

```xml
<tag attribute="value" id="unique-name" class="list of classes">
  outer content
  <child attribute="value" id="unique-name" class="list of classes">
    inner content
  </child>
  more content
</tag>
```

We can use `HTML`/`XML` _selectors_ to directly filter for elements with particular properties.

### Selectors

- `tag`: select by tag name
- `#id`: select by id
- `.class`: select by class

- `[attribute]`: select all elements with an attribute
- `[attribute='value']`: select all elements with an attribute with a given value

- `sel1,sel2`: select all sel1 and all sel2 elements
- `sel1 sel2`: select all sel2 elements that are children of sel1
- `sel1 > sel2`: select all sel2 elements that are direct children of sel1

## Beautiful Soup

The `Beautiful Soup` library gives us a pretty easy way to do exactly this.

We just have to give it a `html` or `xml` string, and then we can use the `select()` function to traverse the file elements.

In [ ]:
res = requests.get("https://rss.nytimes.com/services/xml/rss/nyt/US.xml")
nyt_txt = res.text
nyt_soup = BeautifulSoup(nyt_txt, features="xml")

### Get all `<item>` tags

In [ ]:
nyt_soup.select("item")

In [ ]:
for item in nyt_soup.select("item"):
  item_title = item.select("title")[0]
  item_date = item.select("pubDate")[0]
  item_media = item.select("media|content")
  print(item_title.text)
  print(item_date.text)
  if len(item_media) > 0:
    print(item_media[0]["url"])

### Get all `<title>` tags that are children of `<item>` tags

In [ ]:
for title in nyt_soup.select("item title"):
  print(title)

### Get all `<item>` tags, then get its `<link>`, `<title>` and `<media>` children

This: 
```py
item.select("media|content[medium='image']")
```

selects every `<media:content>` tag inside `item` that has a `medium` attribute set to `image`.

Something like this:

```xml
<item>
  ...
  <media:content width="800" height="450" medium="image" url="https://static01.nyt.com/images/file/jpg" />
  ...
</item>
```

In [ ]:
for item in nyt_soup.select("item"):
  title = item.select("title")
  link = item.select("link")
  media = item.select("media|content[medium='image']")
  if len(media) > 0:
    print(f"{title[0].text}\n {link[0].text}\n {media[0]['url']}\n")

### With `html`

We can go directly to the elements that have the content we are interested in.

In the case of the `https://www1.folha.uol.com.br/ambiente/` page, we're looking for:
- all of the `<div>` elements that belong to the `.c-headline` class

<br>Inside of those:
- the title is in an `<h2>` tag inside a `<div>` of class `.c-headline__content`
- the article url is in an `<a>` tag inside a `<div>` of class `.c-headline__content`
- the image url is in an `<img>` tag inside a `<div>` of class `.c-headline__media-wrapper`

In [ ]:
res = requests.get("https://www1.folha.uol.com.br/ambiente/")
res.encoding = "utf8"
fsp_climate = res.text
fsp_soup = BeautifulSoup(fsp_climate, "html.parser")

In [ ]:
fsp_headlines = fsp_soup.select(".c-headline")

articles = []

for headline in fsp_headlines:
  title_el = headline.select(".c-headline__content h2")
  link_el = headline.select(".c-headline__content a")
  img_el = headline.select(".c-headline__media-wrapper img")
  if len(title_el) > 0:
    articles.append({
      "title": title_el[0].text.strip(),
      "link": link_el[0]['href'],
      "image": img_el[0]['data-src'],
    })

display(len(articles))
display(articles[:8])

### Yeah !

Much easier than plain RegEx for deeply nested `html`/`xml` files.